# `RegistryBase`

`nematics3d.RegistryBase` is a lightweight ordered registry for named objects. It is useful when a `Nematics3D` object owns a collection of other named objects and those objects need stable lookup, insertion order, unique names, and a `registry` relation back to the collection that currently owns them.

The most common operations are `act_register()`, lookup by name or index, iteration, `act_unregister()`, and `act_clear()`. Duplicate names are resolved automatically by adding a numeric suffix.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** Run the following cell to import `NumPy` and `Nematics3D`, and to build a small helper function that creates `SmoothedLine` objects for the registry examples.


In [ ]:
import numpy as np
import nematics3d as n3d


def make_line(name, phase=0.0):
    x = np.linspace(0.0, 2.0 * np.pi, 61)
    coords = np.column_stack([x, np.sin(x + phase), np.zeros_like(x)])
    return n3d.SmoothedLine(
        coords,
        name=name,
        window_length=7,
        min_line_length=2,
    )


## Minimal example: register and retrieve named objects

Create a registry, register two objects, and retrieve them either by name or by insertion-order index. `act_register()` returns the registered object, so construction and registration can be combined in one expression.


In [ ]:
registry = n3d.RegistryBase("example lines")
first = registry.act_register(make_line("first"))
second = registry.act_register(make_line("second", phase=0.5))

print(registry["first"] is first)
print(registry[1] is second)
print(len(registry))


## Public interface

The main collection interface is:

| Operation | Meaning |
| --- | --- |
| `registry.act_register(obj)` | Register one named object. If another registered object already has the same name, the new object is renamed automatically. The registered object is returned. |
| `registry[name]` | Retrieve an object by its current registered name. |
| `registry[index]` | Retrieve an object by insertion-order index. |
| `len(registry)` | Return the number of registered objects. |
| `for obj in registry` | Iterate over objects in insertion order. |
| `registry.entity` | Return the registered objects as an immutable tuple. |
| `registry.act_unregister(obj)` | Remove one object and unbind its `registry` relation when that relation points to this registry. |
| `registry.act_clear()` | Remove all registered objects. |

`repr(registry)` shows the registry contents in order, while `str(registry)` keeps the compact identity-style representation used by the common `ClassBase` object model.


## Insertion order and the read-only collection view

Iteration preserves registration order. The `entity` property exposes the same objects as a tuple, so callers can inspect the collection without receiving the mutable internal list.


In [ ]:
print([obj.name for obj in registry])
print(type(registry.entity))
print(registry.entity == (first, second))


## Names remain unique inside one registry

If a newly registered object already has a name used by another object, `RegistryBase` appends `_1`, `_2`, and so on until the name is unique.


In [ ]:
duplicate = registry.act_register(make_line("first", phase=1.0))
print(duplicate.name)
print([obj.name for obj in registry])


A registered `ClassBase` object also respects the registry's uniqueness rule when it is renamed. Reassigning its current name leaves it unchanged; requesting another object's name receives the next available suffix.


In [ ]:
first.act_set_name("first")
print(first.name)

second.act_set_name("first")
print(second.name)
print([obj.name for obj in registry])


## The `registry` relation and moving an object

For normal `ClassBase` objects, registration also binds a weak `registry` relation. Registering the same object in a different registry moves it: the old registry removes it first, then the new registry becomes the object's current `registry`.


In [ ]:
other = n3d.RegistryBase("other lines")
other.act_register(first)

print(first.registry is other)
print(first in other)
print(first in registry)


## Removing objects

`act_unregister()` removes one object. `act_clear()` removes the complete collection. When `is_return_removed=True`, `act_clear()` returns the removed objects as a tuple in their previous registry order.


In [ ]:
other.act_unregister(first)
print(first.registry)
print(len(other))

removed = registry.act_clear(
    is_return_removed=True,
    is_show_existing=False,
)
print([obj.name for obj in removed])
print(len(registry))


## Summary

Use `RegistryBase` when a group of named `Nematics3D` objects should behave like one ordered collection. Registration preserves insertion order, enforces unique names, supports lookup by name or index, and keeps the registered object's `registry` relation synchronized when the normal `ClassBase` relation interface is available.
